# Get the Audio Files

In [ ]:
!pip install -U yt-dlp
!apt-get update && apt-get install ffmpeg nodejs -y

In [ ]:
import subprocess
import os

# The 35-minute YouTube video link
video_url = "https://www.youtube.com/watch?v=IZaN8DlWZsA"
output_dir = "/content/audio_raw"
os.makedirs(output_dir, exist_ok=True)

print("Initializing high-quality extraction for long-form audio stream...")

command = [
    "yt-dlp",
    "--extract-audio",
    "--audio-format", "mp3",
    "--audio-quality", "320K",             # Force highest constant bitrate (320kbps)
    "--format", "bestaudio",
    "--ignore-errors",
    "--no-warnings",
    # Uses a secure desktop browser identity to slide past length/bot restrictions
    "--extractor-args", "youtube:player_client=web,web_safari;-android_sdkless",
    "--user-agent", "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
    "--output", f"{output_dir}/%(title)s.%(ext)s",
    video_url
]

try:
    result = subprocess.run(command, check=True, text=True, capture_output=True)
    print("\nSuccess! Audio download and MP3 conversion completed.")
    print("Files available in workspace:")
    print(os.listdir(output_dir))
except subprocess.CalledProcessError as e:
    print("\nExtraction Failed:")
    print(e.stderr)

In [ ]:
!yt-dlp \
    --extract-audio \
    --audio-format wav \
    --audio-quality 0 \
    --format "bestaudio" \
    --cookies /content/cookies.txt \
    --output "/content/documentary_audio/%(title)s.%(ext)s" \
    "https://docuseek2-com.proxy1.library.virginia.edu/if-guns"

#Create the transcripts

In [ ]:
!pip install whisperx

# Separate steps

In [ ]:
import whisperx
import gc
import torch
from google.colab import userdata

# 1. Configuration variables
device = "cuda" if torch.cuda.is_available() else "cpu"
audio_file = "/content/UsKids.mp3"
batch_size = 16          # Reduce if you run out of GPU VRAM
compute_type = "float16" # Use float16 for fast execution on Colab GPU

# Automatically grab your token securely from Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    print("Warning: Could not find HF_TOKEN in Colab Secrets. Falling back to manual variable assignment.")
    hf_token = "YOUR_HUGGINGFACE_ACCESS_TOKEN" # Fallback if you prefer to hardcode it

# 2. Transcribe audio with WhisperX
print("Loading WhisperX model...")
model = whisperx.load_model("large-v3", device, compute_type=compute_type)

print("Transcribing audio...")
audio = whisperx.load_audio(audio_file)
result = model.transcribe(audio, batch_size=batch_size)

# Clear VRAM to prevent crash before diarization
model = None
gc.collect()
torch.cuda.empty_cache()

# 3. Align Whisper timestamps to a word-level grid
print("Aligning timestamps...")
model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

# Clear VRAM again
model_a = None
gc.collect()
torch.cuda.empty_cache()



In [ ]:
# 4. Run Pyannote Diarization using the correct internal path
print("Loading Pyannote diarization pipeline...")

# FIX: Use the newer 'token' parameter instead of the deprecated 'use_auth_token'
diarize_model = whisperx.diarize.DiarizationPipeline(token=hf_token, device=device)

print("Assigning speakers to text...")
diarize_segments = diarize_model(audio)
result = whisperx.assign_word_speakers(diarize_segments, result)

In [ ]:
# 5. Format, Print, and Save the final transcript
output_filename = "UsKids.txt"
print(f"\n--- Printing and Saving Transcript to {output_filename} ---")

with open(output_filename, "w", encoding="utf-8") as f:
    for segment in result["segments"]:
        start_time = segment["start"]
        speaker = segment.get("speaker", "UNKNOWN_SPEAKER")
        text = segment["text"].strip()

        # Format time into clean MM:SS
        minutes = int(start_time // 60)
        seconds = int(start_time % 60)
        timestamp = f"[{minutes:02d}:{seconds:02d}]"

        # Format the line
        line = f"{timestamp} {speaker}: {text}\n"

        # Print to console and write to file
        print(line, end="")
        f.write(line)

print(f"\nSaved successfully! You can download '{output_filename}' from the Colab file sidebar.")

# All in one

In [1]:
%pip install git+https://github.com/m-bain/whisperX.git

  Cloning https://github.com/m-bain/whisperX.git to /tmp/pip-req-build-zkx5rsiz
  Running command git clone --filter=blob:none --quiet https://github.com/m-bain/whisperX.git /tmp/pip-req-build-zkx5rsiz
  Resolved https://github.com/m-bain/whisperX.git to commit 2cfd7b7c5c7bba144954364db747319b50e8232b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

# Grab the token from your terminal or paste it here for this notebook session
os.environ["HF_TOKEN"] = "HF_TOKEN"

print("✓ HF_TOKEN is active:", bool(os.environ.get("HF_TOKEN")))

✓ HF_TOKEN is active: True


In [3]:
import os
import gc
import torch
import whisperx
from pathlib import Path

# 1. Configuration & Global Variables
device = "cuda" if torch.cuda.is_available() else "cpu"

# High-performance precision for NVIDIA GPUs (A100/V100/L40/T4)
compute_type = "float16" if device == "cuda" else "int8"

# Point to your actual scratch directory on HPC
input_dir = Path("/scratch/pkb5nz/raw_audio")  # or Path("./get_audio")

# Retrieve HF Token from environment or set it explicitly here if needed
hf_token = os.environ.get("HF_TOKEN", "YOUR_HUGGINGFACE_ACCESS_TOKEN")

print("--- Phase 1: Pre-loading Neural Network Models into GPU VRAM ---")
if device == "cuda":
    print(f"Running on GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ CUDA not detected! Running on CPU.")

print("Loading primary WhisperX model (large-v3)...")
whisper_model = whisperx.load_model("large-v3", device, compute_type=compute_type)

print("Loading Pyannote diarization pipeline...")
diarize_model = whisperx.diarize.DiarizationPipeline(token=hf_token, device=device)

# Dictionary to cache alignment models dynamically so we don't reload them
alignment_model_cache = {}

print("\n✓ Both models are fully loaded into GPU memory and ready to process files!")

--- Phase 1: Pre-loading Neural Network Models into GPU VRAM ---
Running on GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Loading primary WhisperX model (large-v3)...


/home/pkb5nz/.conda/envs/bertopic/lib/python3.10/site-packages/pyannote/audio/core/io.py:48: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6 and 7.
          2. The PyTorch version (2.8.0+cu128) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.
        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 7: libavut

2026-08-03 08:24:55 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-08-03 08:24:55 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../home/pkb5nz/.conda/envs/bertopic/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


Loading Pyannote diarization pipeline...
2026-08-03 08:24:55 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1

✓ Both models are fully loaded into GPU memory and ready to process files!


In [9]:
%pip install static-ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 634.7/634.7 kB 42.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.7/806.7 kB 69.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 113.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22/22 [static-ffmpeg]0m [static-ffmpeg]
Note: you may need to restart the kernel to use updated packages.


In [10]:
import static_ffmpeg
import os
import shutil

# Download/bundle standalone static binaries (no system .so files required)
static_ffmpeg.add_paths()

print("Active ffmpeg binary location:", shutil.which("ffmpeg"))

# Test execution
import subprocess
res = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
print("✓ Static ffmpeg working!" if res.returncode == 0 else f"❌ Error: {res.stderr}")


Download of https://github.com/zackees/ffmpeg_bins/raw/main/v8.0/linux.zip -> /home/pkb5nz/.conda/envs/bertopic/lib/python3.10/site-packages/static_ffmpeg/bin/linux.zip completed.
Extracting /home/pkb5nz/.conda/envs/bertopic/lib/python3.10/site-packages/static_ffmpeg/bin/linux.zip -> /home/pkb5nz/.conda/envs/bertopic/lib/python3.10/site-packages/static_ffmpeg/bin
Active ffmpeg binary location: /home/pkb5nz/.conda/envs/bertopic/lib/python3.10/site-packages/static_ffmpeg/bin/linux/ffmpeg
✓ Static ffmpeg working!


In [11]:
# Ensure the models exist in the current session notebook memory
if 'whisper_model' not in locals() or 'diarize_model' not in locals():
    raise NameError("Models not found in memory. Please run Cell 1 first to load them!")

batch_size = 16  # Reduce to 8 if you encounter VRAM limitations on very long files

# Scan folder for supported file tracks
supported_extensions = ('.mp3', '.wav', '.m4a', '.flac')
audio_files = sorted([
    f for f in os.listdir(input_dir)
    if f.lower().endswith(supported_extensions)
])

if not audio_files:
    print(f"No supported audio files found in {input_dir}. Please upload files to your sidebar.")
else:
    print(f"Found {len(audio_files)} file(s) to process: {audio_files}\n")
    print("=" * 60)

    for index, filename in enumerate(audio_files, 1):
        full_audio_path = os.path.join(input_dir, filename)

        # Determine output text name: "file.mp3" -> "file.txt"
        base_name, _ = os.path.splitext(filename)
        output_filename = os.path.join(input_dir, f"{base_name}.txt")

        print(f"\nProcessing file [{index}/{len(audio_files)}]: {filename}")
        print(f"Output destination: {output_filename}")
        print("-" * 40)

        try:
            # A. Load audio matrix into memory
            print("-> Loading audio track into memory matrix...")
            audio = whisperx.load_audio(full_audio_path)

            # B. Transcribe Core Text
            print("-> Executing forward-pass transcription...")
            result = whisper_model.transcribe(audio, batch_size=batch_size, language="en") # <--- Force English here
            language_code = result["language"]
            print(f"   Forced/Detected Language: {language_code}")

            # C. Dynamic Timestamp Alignment
            print("-> Structuring word-level timestamp alignment...")
            if language_code not in alignment_model_cache:
                model_a, metadata = whisperx.load_align_model(language_code=language_code, device=device)
                alignment_model_cache[language_code] = (model_a, metadata)
            else:
                model_a, metadata = alignment_model_cache[language_code]

            result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

            # D. Speaker Diarization
            print("-> Running speaker vocal diarization partitions...")
            diarize_segments = diarize_model(audio)
            result = whisperx.assign_word_speakers(diarize_segments, result)

            # E. Format and Save Output directly to file
            print(f"-> Generating log arrays and saving to {output_filename}...")
            with open(output_filename, "w", encoding="utf-8") as f:
                for segment in result["segments"]:
                    start_time = segment.get("start", 0.0)
                    speaker = segment.get("speaker", "UNKNOWN_SPEAKER")
                    text = segment["text"].strip()

                    # Convert seconds seamlessly into clean standard [MM:SS] strings
                    minutes = int(start_time // 60)
                    seconds = int(start_time % 60)
                    timestamp = f"[{minutes:02d}:{seconds:02d}]"

                    line = f"{timestamp} {speaker}: {text}\n"
                    f.write(line)

            print(f"✓ Successfully finalized: {filename}")

        except Exception as e:
            print(f"❌ Error processing {filename}: {str(e)}")
            print("Skipping to next track...")

        # Periodic cache cleanup inside the loop to stabilize CUDA allocations
        gc.collect()
        torch.cuda.empty_cache()

    print("\n" + "=" * 60)
    print("Batch processing completed! All transcripts are compiled in the sidebar.")

Found 4 file(s) to process: [' Chi-Town Guns, Gun Violence and The NRA.mp3', 'AlwaysInSeason.mp3', 'Cities in Crisis - Youth Violence.mp3', 'Real Life Teens - Guns at School - How Safe Do Teens Feel.mp3']


Processing file [1/4]:  Chi-Town Guns, Gun Violence and The NRA.mp3
Output destination: /scratch/pkb5nz/raw_audio/ Chi-Town Guns, Gun Violence and The NRA.txt
----------------------------------------
-> Loading audio track into memory matrix...
-> Executing forward-pass transcription...


/home/pkb5nz/.conda/envs/bertopic/lib/python3.10/site-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(


   Forced/Detected Language: en
-> Structuring word-level timestamp alignment...
Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth" to /home/pkb5nz/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960_asr_ls960.pth


100%|██████████| 360M/360M [00:01<00:00, 347MB/s] 


2026-08-03 08:29:06 - whisperx.alignment - INFO - Downloading NLTK punkt_tab data for sentence splitting...
-> Running speaker vocal diarization partitions...


/home/pkb5nz/.conda/envs/bertopic/lib/python3.10/site-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


-> Generating log arrays and saving to /scratch/pkb5nz/raw_audio/ Chi-Town Guns, Gun Violence and The NRA.txt...
✓ Successfully finalized:  Chi-Town Guns, Gun Violence and The NRA.mp3

Processing file [2/4]: AlwaysInSeason.mp3
Output destination: /scratch/pkb5nz/raw_audio/AlwaysInSeason.txt
----------------------------------------
-> Loading audio track into memory matrix...
-> Executing forward-pass transcription...
   Forced/Detected Language: en
-> Structuring word-level timestamp alignment...
-> Running speaker vocal diarization partitions...
-> Generating log arrays and saving to /scratch/pkb5nz/raw_audio/AlwaysInSeason.txt...
✓ Successfully finalized: AlwaysInSeason.mp3

Processing file [3/4]: Cities in Crisis - Youth Violence.mp3
Output destination: /scratch/pkb5nz/raw_audio/Cities in Crisis - Youth Violence.txt
----------------------------------------
-> Loading audio track into memory matrix...
-> Executing forward-pass transcription...
   Forced/Detected Language: en
-> Struc